# Camera Discovery Live Test

Simplified 3-stage pipeline: `TargetResolver → CandidateDiscoveryEngine → ReviewAndValidationPipeline`.

LLMs are used as advisory evidence interpreters/rankers for target intent, geocoder candidate ranking, and candidate semantic review. Deterministic code/tools remain responsible for geometry verification, stream validation, trusted-output authorization, and final artifact writing.

This notebook clones the `main` branch of the GitHub repository by default, installs it in editable mode, runs a live test, and then displays trusted or untrusted camera outputs.

This notebook supports single-location and multi-location queries, for example:

```text
Get me all traffic cameras from California
Get me all cameras from Greenville, Texas
Get me all cameras from London, England and New York, New York
```

The query is intentionally user-controlled. Phrases such as `traffic cameras`, `weather cameras`, or `public live cameras` should be interpreted as camera-type intent, while place names such as `California`, `Greenville, Texas`, or `London, England` are target geography.


In [1]:
from pathlib import Path
import os
import sys
import subprocess
import json
import shutil

# Colab/repo bootstrap settings. Override with env vars if needed.
REPO_URL = os.environ.get("CAMERA_DISCOVERY_REPO_URL", "https://github.com/dshipley71/camera-discovery.git")
REPO_BRANCH = os.environ.get("CAMERA_DISCOVERY_REPO_BRANCH", "main")
REPO_DIR = Path(os.environ.get("CAMERA_DISCOVERY_REPO_DIR", "/content/camera-discovery"))

print("Notebook bootstrap")
print("repo url:", REPO_URL)
print("branch:", REPO_BRANCH)
print("repo dir:", REPO_DIR)


Notebook bootstrap
repo url: https://github.com/dshipley71/camera-discovery.git
branch: main
repo dir: /content/camera-discovery


In [2]:
%cd /content

if REPO_DIR.exists():
    print(f"Removing existing repo directory: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

clone_cmd = ["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)]
print("$", " ".join(clone_cmd))
subprocess.run(clone_cmd, check=True)

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

src_path = REPO_DIR / "src"
assert (src_path / "camera_discovery").exists(), f"Missing package at {src_path / 'camera_discovery'}"

# Make imports work immediately, even before editable install finishes.
os.environ["PYTHONPATH"] = str(src_path)
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

install_cmd = [sys.executable, "-m", "pip", "install", "-e", ".", "--no-build-isolation"]
print("$", " ".join(install_cmd))
subprocess.run(install_cmd, check=True)


/content
$ git clone -b main https://github.com/dshipley71/camera-discovery.git /content/camera-discovery
cwd: /content/camera-discovery
$ /usr/bin/python3 -m pip install -e . --no-build-isolation


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-e', '.', '--no-build-isolation'], returncode=0)

In [3]:
import camera_discovery
print("camera_discovery import OK:", camera_discovery.__file__)

# Load provider secrets from Colab userdata when available.
# Configure these in Colab as needed:
#   OLLAMA_API_KEY
#   OPENAI_API_KEY
#   OPENAI_BASE_URL
#   AWS_ACCESS_KEY_ID
#   AWS_SECRET_ACCESS_KEY
#   AWS_SESSION_TOKEN
#   AWS_DEFAULT_REGION
try:
    from google.colab import userdata  # type: ignore
except Exception as exc:
    userdata = None
    print("Colab userdata not available:", repr(exc))

if userdata is not None:
    for key in [
        "OLLAMA_API_KEY",
        "OPENAI_API_KEY",
        "OPENAI_BASE_URL",
        "AWS_ACCESS_KEY_ID",
        "AWS_SECRET_ACCESS_KEY",
        "AWS_SESSION_TOKEN",
        "AWS_DEFAULT_REGION",
    ]:
        if os.environ.get(key):
            continue
        try:
            value = userdata.get(key)
        except Exception:
            value = None
        if value:
            os.environ[key] = value
            print(f"Loaded {key} from Colab userdata")

!python -m camera_discovery.cli --help
!python -m camera_discovery.cli run --help


camera_discovery import OK: /content/camera-discovery/src/camera_discovery/__init__.py
Loaded OLLAMA_API_KEY from Colab userdata
Loaded OPENAI_API_KEY from Colab userdata
                                                                                
 Usage: python -m camera_discovery.cli [OPTIONS] COMMAND [ARGS]...              
                                                                                
 Simplified public camera discovery pipeline                                    
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --install-completion          Install completion for the current shell.      │
│ --show-completion             Show completion for the current shell, to copy │
│                               it or customize the installation.              │
│ --help                        Show this message and exit.                    │
╰──────────────────

| Profile    | Purpose                    | Behavior                                                                                                                                                                             |
| ---------- | -------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `fast`     | Quick review/discovery run | Validation is minimized/disabled; trusted `camera.geojson` should not be produced unless trust requirements are met; useful for `untrusted_camera_candidates.geojson` review output. |
| `balanced` | Middle-ground run          | More validation than Fast, but avoids the most expensive checks. Good default for routine testing.                                                                                   |
| `full`     | Most thorough run          | Runs the deepest validation path available, intended for trusted output when geometry and stream validation pass.                                                                    |


In [4]:
RUN_PROFILE = os.environ.get("CAMERA_DISCOVERY_PROFILE", "fast").strip().lower()
if RUN_PROFILE not in {"fast", "balanced", "full"}:
    raise ValueError(f"Invalid CAMERA_DISCOVERY_PROFILE={RUN_PROFILE!r}; expected fast, balanced, or full")

# User-controlled query. Edit this directly or set CAMERA_DISCOVERY_QUERY in the environment.
# Camera-type terms such as "traffic cameras" are intentional camera intent, not target geography.
USER_QUERY = os.environ.get("CAMERA_DISCOVERY_QUERY", "Get me all traffic cameras from California")

# Other examples:
# USER_QUERY = "Get me all cameras from Greenville, Texas"
# USER_QUERY = "Get me all cameras from London, England and New York, New York"

OUTPUT_DIR = Path(os.environ.get("CAMERA_DISCOVERY_OUTPUT_DIR", "runs/notebook-live-test"))
CLEAN_OUTPUT_DIR = os.environ.get("CAMERA_DISCOVERY_CLEAN_OUTPUT_DIR", "true").strip().lower() in {"1", "true", "yes", "on"}

# LLM provider defaults. This application requires a real LLM provider.
os.environ.setdefault("CAMERA_DISCOVERY_LLM_PROVIDER", "ollama-cloud")
os.environ.setdefault("CAMERA_DISCOVERY_LLM_MODEL", "gemma3:4b-cloud")
os.environ.setdefault("CAMERA_DISCOVERY_TARGET_INTENT_MODEL", "gemma3:4b-cloud")
os.environ.setdefault("CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL", "gemma3:4b-cloud")
os.environ.setdefault("CAMERA_DISCOVERY_CANDIDATE_REVIEW_MODEL", "gemma3:4b-cloud")

DISCOVERY_MODE = os.environ.get("CAMERA_DISCOVERY_DISCOVERY_MODE", "both").strip().lower()
if DISCOVERY_MODE not in {"blind", "directory", "both", "direct"}:
    raise ValueError(f"Invalid CAMERA_DISCOVERY_DISCOVERY_MODE={DISCOVERY_MODE!r}")

SOURCES_FILE = Path(os.environ.get("CAMERA_DISCOVERY_SOURCES_FILE", "SOURCES.md"))
SEED_URLS = [url.strip() for url in os.environ.get("CAMERA_DISCOVERY_SEED_URLS", "").split(",") if url.strip()]

provider = os.environ.get("CAMERA_DISCOVERY_LLM_PROVIDER", "").strip()
required_secret_hint = {
    "ollama": "OLLAMA_API_KEY is required for Ollama Cloud; local Ollama may not need it.",
    "ollama-cloud": "OLLAMA_API_KEY is required.",
    "openai-compatible": "OPENAI_API_KEY and OPENAI_BASE_URL are usually required.",
    "bedrock": "AWS credentials and AWS_DEFAULT_REGION are required.",
}.get(provider, "provider-specific credentials are required")

print("profile:", RUN_PROFILE)
print("query:", USER_QUERY)
print("output:", OUTPUT_DIR)
print("clean output dir before run:", CLEAN_OUTPUT_DIR)
print("provider:", provider)
print("model:", os.environ.get("CAMERA_DISCOVERY_LLM_MODEL"))
print("target intent model:", os.environ.get("CAMERA_DISCOVERY_TARGET_INTENT_MODEL"))
print("geocoder referee model:", os.environ.get("CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL"))
print("candidate review model:", os.environ.get("CAMERA_DISCOVERY_CANDIDATE_REVIEW_MODEL"))
print("discovery mode:", DISCOVERY_MODE)
print("sources file:", SOURCES_FILE, "exists=", SOURCES_FILE.exists())
print("seed urls:", len(SEED_URLS))
print("credential hint:", required_secret_hint)

if DISCOVERY_MODE in {"directory", "both"} and not SOURCES_FILE.exists():
    print(f"WARNING: {SOURCES_FILE} does not exist. Directory sources will be empty unless the repo provides it.")
if DISCOVERY_MODE == "direct" and not SEED_URLS:
    raise ValueError("DISCOVERY_MODE=direct requires CAMERA_DISCOVERY_SEED_URLS or --seed-url values")


profile: fast
query: Get me all traffic cameras from California
output: runs/notebook-live-test
clean output dir before run: True
provider: ollama-cloud
model: gemma3:4b-cloud
target intent model: gemma3:4b-cloud
geocoder referee model: gemma3:4b-cloud
candidate review model: gemma3:4b-cloud
discovery mode: both
sources file: SOURCES.md exists= True
seed urls: 0
credential hint: OLLAMA_API_KEY is required.


In [5]:
cmd = [
    sys.executable, "-m", "camera_discovery.cli", "run", USER_QUERY,
    "--profile", RUN_PROFILE,
    "--output-dir", str(OUTPUT_DIR),
    "--discovery-mode", DISCOVERY_MODE,
    "--sources-file", str(SOURCES_FILE),
]
for url in SEED_URLS:
    cmd.extend(["--seed-url", url])

# Remove stale run artifacts before each live test unless explicitly disabled.
if CLEAN_OUTPUT_DIR and OUTPUT_DIR.exists():
    resolved_output = OUTPUT_DIR.resolve()
    resolved_repo = REPO_DIR.resolve()
    unsafe_roots = {Path("/").resolve(), Path("/content").resolve(), resolved_repo}
    if resolved_output in unsafe_roots:
        raise RuntimeError(f"Refusing to remove unsafe output directory: {resolved_output}")
    print(f"Removing stale output directory: {resolved_output}")
    shutil.rmtree(resolved_output)

print("$", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)

run_stdout = OUTPUT_DIR / "notebook_cli_stdout.log"
run_stderr = OUTPUT_DIR / "notebook_cli_stderr.log"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_stdout.write_text(result.stdout or "", encoding="utf-8")
run_stderr.write_text(result.stderr or "", encoding="utf-8")

print(result.stdout)
print(result.stderr)
print("exit:", result.returncode)
print("stdout log:", run_stdout)
print("stderr log:", run_stderr)

if result.returncode != 0:
    raise RuntimeError("camera-discovery run failed; inspect notebook_cli_stdout.log and notebook_cli_stderr.log")


$ /usr/bin/python3 -m camera_discovery.cli run Get me all traffic cameras from California --profile fast --output-dir runs/notebook-live-test --discovery-mode both --sources-file SOURCES.md
Profile: fast
Validation enabled: False
LLM provider: ollama-cloud
Target-intent model: gemma3:4b-cloud
Discovery mode: both
Sources file: SOURCES.md
Targets resolved: 1
  - traffic_cameras: traffic_cameras | geometry=missing; verified=False; 
policy=review_only
Discovering: traffic_cameras
Candidates: raw=0 unique=0 coordinate_bearing=0 targets=1
Trusted GeoJSON: False features=0
Untrusted GeoJSON: False features=0
Review package: runs/notebook-live-test/review_artifacts.zip


exit: 0
stdout log: runs/notebook-live-test/notebook_cli_stdout.log
stderr log: runs/notebook-live-test/notebook_cli_stderr.log


In [6]:
from pathlib import Path
import json

for rel in ["logs/source_policy_summary.json", "logs/candidate_discovery_summary.json", "logs/run_summary.json"]:
    path = OUTPUT_DIR / rel
    print("---", rel, "exists=", path.exists())
    if path.exists():
        try:
            print(json.dumps(json.loads(path.read_text(encoding="utf-8")), indent=2)[:4000])
        except Exception as exc:
            print("Could not parse JSON:", repr(exc))
            print(path.read_text(encoding="utf-8")[:1000])


--- logs/source_policy_summary.json exists= True
{
  "allowed_sources": [],
  "blocked_sources": [],
  "enabled_allowed_sources": 0,
  "source_file": "SOURCES.md"
}
--- logs/candidate_discovery_summary.json exists= True
{
  "allowed_directory_sources": 0,
  "blocked_patterns": 0,
  "coordinate_bearing": 0,
  "discovery_mode": "both",
  "in_scope": 0,
  "llm_semantic_reviewed": 0,
  "raw": 0,
  "rejected": 0,
  "review": 0,
  "sources_file": "SOURCES.md",
  "target_id": "traffic_cameras",
  "target_label": "traffic_cameras",
  "unique": 0
}
--- logs/run_summary.json exists= True
{
  "candidate_sets_by_target": {
    "traffic_cameras": {
      "coordinate_bearing": [],
      "in_scope": [],
      "raw": [],
      "rejected": [],
      "review": [],
      "unique": []
    }
  },
  "candidates": {
    "coordinate_bearing": [],
    "in_scope": [],
    "raw": [],
    "rejected": [],
    "review": [],
    "unique": []
  },
  "config": {
    "allow_untrusted_review_output": true,
    "block_pa

In [7]:
summary_path = OUTPUT_DIR / "logs" / "run_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
targets = summary.get("targets", [])
print("targets:", len(targets))
for t in targets:
    print(json.dumps({
        "target_id": t.get("target_id"),
        "target_label": t.get("target_label"),
        "canonical_target": t.get("canonical_target"),
        "geometry_status": t.get("geometry_status"),
        "bbox_verified": t.get("bbox_verified"),
        "trust_policy": t.get("trust_policy"),
    }, indent=2))

candidate_summary = summary.get("candidates", {}) or {}
output_summary = summary.get("outputs", {}) or {}
unique_value = candidate_summary.get("unique_count")
if unique_value is None and isinstance(candidate_summary.get("unique"), list):
    unique_value = len(candidate_summary.get("unique"))
coord_value = candidate_summary.get("coordinate_bearing_count")
if coord_value is None and isinstance(candidate_summary.get("coordinate_bearing"), list):
    coord_value = len(candidate_summary.get("coordinate_bearing"))

print(json.dumps({
    "unique_candidates": unique_value,
    "coordinate_bearing": coord_value,
    "trusted_geojson_features": output_summary.get("trusted_geojson_features_written", 0),
    "untrusted_geojson_features": output_summary.get("untrusted_geojson_features_written", 0),
    "trusted_geojson_created": output_summary.get("trusted_geojson_created"),
    "untrusted_geojson_created": output_summary.get("untrusted_geojson_created"),
}, indent=2))


targets: 1
{
  "target_id": "traffic_cameras",
  "target_label": "traffic_cameras",
  "canonical_target": "traffic_cameras",
  "geometry_status": "missing",
  "bbox_verified": false,
  "trust_policy": "review_only"
}
{
  "unique_candidates": 0,
  "coordinate_bearing": 0,
  "trusted_geojson_features": 0,
  "untrusted_geojson_features": 0,
  "trusted_geojson_created": false,
  "untrusted_geojson_created": false
}


In [8]:
for rel in [
    'camera.geojson',
    'untrusted_camera_candidates.geojson',
    'map.html',
    'review_artifacts.zip',
    'logs/target_resolution_all.json',
    'logs/target_intent.json',
    'logs/geocoder_referee.json',
    'logs/candidate_semantic_review.json',
    'logs/geocoder_candidate_scores.json',
    'logs/output_summary.json',
]:
    p = OUTPUT_DIR / rel
    print(rel, 'exists=', p.exists(), 'size=', p.stat().st_size if p.exists() else 0)

# Per-target diagnostics are written below logs/targets/<target_id>/ and candidates/<target_id>/.
for folder in sorted((OUTPUT_DIR / 'logs' / 'targets').glob('*')) if (OUTPUT_DIR / 'logs' / 'targets').exists() else []:
    print('target diagnostics:', folder.relative_to(OUTPUT_DIR))


camera.geojson exists= False size= 0
untrusted_camera_candidates.geojson exists= False size= 0
map.html exists= True size= 7404
review_artifacts.zip exists= True size= 717591
logs/target_resolution_all.json exists= True size= 567
logs/target_intent.json exists= True size= 662
logs/geocoder_referee.json exists= False size= 0
logs/candidate_semantic_review.json exists= False size= 0
logs/geocoder_candidate_scores.json exists= True size= 3379443
logs/output_summary.json exists= True size= 212
target diagnostics: logs/targets/traffic_cameras


## Camera URL table

This cell loads `camera.geojson` first, then falls back to `untrusted_camera_candidates.geojson` or `untrusted_camera.geojson`. It displays URL, location, coordinate, trust, validation, and source metadata in a table and writes a CSV copy into the run directory.

In [9]:
from pathlib import Path
from IPython.display import display
from camera_discovery.utils.geojson_viewer import (
    load_camera_rows,
    select_camera_geojson,
    write_camera_table_csv,
)

GEOJSON_PATH = select_camera_geojson(OUTPUT_DIR)
print("Selected GeoJSON:", GEOJSON_PATH)

if GEOJSON_PATH is None:
    CAMERA_ROWS = []
    print("No trusted or untrusted camera GeoJSON found yet.")
else:
    CAMERA_ROWS = load_camera_rows(GEOJSON_PATH)
    table_csv = write_camera_table_csv(OUTPUT_DIR, CAMERA_ROWS)
    print("Rows:", len(CAMERA_ROWS))
    print("CSV table:", table_csv)
    if not CAMERA_ROWS:
        print("GeoJSON exists but contains no camera features.")
    else:
        try:
            import pandas as pd
            columns = [
                "name", "target_label", "location_text", "latitude", "longitude",
                "stream_url", "source_url", "thumbnail_url", "trust_level",
                "validation_status", "scope_status", "discovery_method", "review_required"
            ]
            df = pd.DataFrame(CAMERA_ROWS)
            display(df[[c for c in columns if c in df.columns]])
        except Exception as exc:
            print("Pandas display unavailable; showing first rows as dictionaries:", repr(exc))
            for row in CAMERA_ROWS[:10]:
                print(row)


Selected GeoJSON: None
No trusted or untrusted camera GeoJSON found yet.


## Interactive camera map

The map below embeds the selected GeoJSON directly into the HTML so it works inside Colab. Click a marker to see camera metadata. If a thumbnail/snapshot URL is present in the GeoJSON properties, the popup shows it. The **Play video** button attempts to play the stream URL with hls.js or native browser video support.

In [10]:
from IPython.display import HTML, display
from camera_discovery.utils.geojson_viewer import write_embedded_camera_map

if GEOJSON_PATH is None:
    print("No GeoJSON available for map display yet.")
else:
    MAP_PATH = write_embedded_camera_map(OUTPUT_DIR, GEOJSON_PATH, output_name="notebook_camera_map.html")
    print("Notebook map:", MAP_PATH)
    display(HTML(MAP_PATH.read_text(encoding="utf-8")))


No GeoJSON available for map display yet.
